# 1 load package and data`

In [ ]:
import json
from collections import namedtuple, Counter

import numpy as np
import pandas as pd
import scanpy as sc
import pandas as pd
import squidpy as sq

## Image manipulation and geometry
from tifffile import imread
from skimage.io import imread as skimread

## Plotting imports
from matplotlib import pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, Normalize, to_hex, Colormap
from matplotlib.cm import ScalarMappable
from matplotlib.colorbar import ColorbarBase
from matplotlib.lines import Line2D
from matplotlib import rc_context

In [ ]:
folder_path = r"P:\PI\PI_Chuva_de_Sousa_Lopes\susana\Fu\Project\20_Xenium 5k embryo\analysis\2st_2025_4_9\backup_adata\QC_10_50_2st\embryo"
h5ad_file = Path(folder_path) / "whole_embryo_hvg_res_1_updata.h5ad"
embryo = sc.read_h5ad(h5ad_file)

# 2 process selected cluster ofextraembryonic tissues 

In [ ]:
import numpy as np
keep = np.array(["2","10","15","5","18","23"], dtype=str)

adata = embryo[embryo.obs["leiden_hvg_1"].isin(keep)].copy()

In [ ]:

if adata.n_obs == 0:
    print("Warning: adata is empty. Aborting processing.")
else:
    print(f"Info: adata selected with {adata.n_obs} cells.")
    

    if 'counts_raw' in adata.layers:
        adata.X = adata.layers['counts_raw'].copy()
        print("Info: .X initialized from 'raw_counts' layer.")
    

    sc.pp.normalize_total(adata, inplace=True)
    sc.pp.log1p(adata)
    

    sc.pp.highly_variable_genes(adata, n_top_genes=2000)

    adata.raw = adata.copy()

    sc.pp.scale(adata, max_value=10)
    

    sc.tl.pca(adata, use_highly_variable=True)
    

    sc.pp.neighbors(adata)
    

    sc.tl.umap(adata, min_dist=0.01, spread=2)
    
    sc.tl.leiden(adata, resolution=0.5, key_added='leiden_sub')
    
    print(f"adata (n={adata.n_obs}) processing complete.")
    print(f"Total genes retained: {adata.n_vars}")
    print(f"Highly variable genes used for analysis: {adata.var.highly_variable.sum()}")

In [ ]:
sc.tl.rank_genes_groups(adata, 
                        groupby="leiden_hvg_sub", 
                        method="wilcoxon", 
                        pts=True,
                        )

In [ ]:
deg_df = sc.get.rank_genes_groups_df(adata, group=None)

deg_df.to_excel(r"P:\PI\PI_Chuva_de_Sousa_Lopes\susana\Fu\Project\20_Xenium 5k embryo\paper\3st_2025_1_27\Figure6\deg\DEG_leiden_hvg_sub_EE.xlsx", index=True)


In [ ]:
sc.tl.dendrogram(adata, groupby="leiden_hvg_sub", linkage_method="ward",cor_method='pearson')  #'pearson', 'spearman', 'kendall',

sc.pl.dendrogram(adata, groupby="leiden_hvg_sub")

In [ ]:
adata.write_h5ad(r"P:\PI\PI_Chuva_de_Sousa_Lopes\susana\Fu\Project\20_Xenium 5k embryo\analysis\2st_2025_4_9\backup_adata\QC_10_50_2st\embryont_extra.h5ad")

# 3  process placenta data

In [ ]:
folder_path = r"P:\PI\PI_Chuva_de_Sousa_Lopes\susana\Fu\Project\20_Xenium 5k embryo\analysis\2st_2025_4_9\backup_adata\QC_10_50_2st"
h5ad_file = Path(folder_path) / "'adata_merge_after_QC_10_50.h5ad"
whole= sc.read_h5ad(h5ad_file)

In [ ]:
placenta = whole[whole.obs['roi_type'] == 'placenta']

In [ ]:
cluster2_subset=placenta.copy()

if cluster2_subset.n_obs == 0:
    print("Warning: cluster2_subset is empty. Aborting processing.")

else:
    print(f"Info: cluster2_subset selected with {cluster2_subset.n_obs} cells.")


    if 'counts_raw' in cluster2_subset.layers:
        cluster2_subset.X = cluster2_subset.layers['counts_raw'].copy()
        print("Info: .X initialized from 'raw_counts' layer.")


    sc.pp.normalize_total(cluster2_subset, inplace=True)
    sc.pp.log1p(cluster2_subset)


    sc.pp.highly_variable_genes(cluster2_subset, n_top_genes=2000)


    cluster2_subset.raw = cluster2_subset.copy() 


    cluster2_subset = cluster2_subset[:, cluster2_subset.var.highly_variable].copy()


    sc.pp.scale(cluster2_subset, max_value=10)
    sc.tl.pca(cluster2_subset)
    sc.pp.neighbors(cluster2_subset)


    sc.tl.umap(cluster2_subset, min_dist=0.01, spread=2)


    sc.tl.leiden(cluster2_subset, resolution=1, key_added='leiden_hvg_1') 

    print(f"cluster2_subset (n={cluster2_subset.n_obs}) processing complete. Sub-clusters stored in 'leiden_sub'.")


In [ ]:
sc.tl.rank_genes_groups(placenta, 
                        groupby="leiden_hvg_1", 
                        method="wilcoxon", 
                        pts=True,
                        )

In [ ]:
rank_genes_df = sc.get.rank_genes_groups_df(placenta, group=None)
rank_genes_df.to_excel('DEG/whole_placenta_deg_hvg_res_1.xlsx', index=True)

In [ ]:
sc.tl.dendrogram(placenta, groupby="leiden_hvg_1", linkage_method="ward",cor_method='pearson')  #'pearson', 'spearman', 'kendall',
sc.pl.dendrogram(placenta, groupby="leiden_hvg_1")

In [ ]:
 placenta.write.h5ad(r"P:\PI\PI_Chuva_de_Sousa_Lopes\susana\Fu\Project\20_Xenium 5k embryo\analysis\2st_2025_4_9\backup_adata\QC_10_50_2st\placenta\whole_placenta_updata_hvg_leiden.h5ad"

# 4 Figure 6c

In [ ]:
custom_palette = {
    "0": "#FF0080",
    "1": "#9400D3",
    "2": "#00E5EE",
    "3": "#ADFF2F",
    "4": "#00FA9A",
    "5": "#FF4500",
    "6": "#FFA500",
    "7": "#F5DEB3",
    "8": "#EE82EE",
    "9": "#1E90FF",
    "10": "#0000CD",
    "11": "#DA70D6",
    "12": "#7FFFD4",
    "13": "#FF69B4",
    "14": "#BA55D3",
    "15": "#DB7093",
    "16": "#DC143C",
    "17": "#FF8C69",
    "18": "#FFC0CB",
    "19": "#FF7F50",
    "20": "#4682B4",
    "21": "#FF00FF",
    "22": "#CD5C5C",
    "23": "#7FFF00",
    "24": "#A9A9A9",
}

In [ ]:

tsne_coords =placenta.obsm['X_umap']


leiden_clusters = placenta.obs['leiden_hvg_1']

import pandas as pd

tsne_df = pd.DataFrame(tsne_coords, columns=['UMAP_1', 'UMAP_2'])
tsne_df['leiden_hvg_1'] = leiden_clusters.values

import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 6))
sns.scatterplot(data=tsne_df, x='UMAP_1', y='UMAP_2', hue='leiden_hvg_1', palette=custom_palette, s=5, edgecolor=None)
plt.title("UMAP Plot with Leiden Clusters")
plt.xlabel("UMAP1")
plt.ylabel("UMAP2")
plt.legend(title="Leiden Cluster", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.savefig(r'P:\PI\PI_Chuva_de_Sousa_Lopes\susana\Fu\Project\20_Xenium 5k embryo\paper\3st_2025_1_27\Figure6\umap\placenta\umap_placenta_newcolor.png', format="png",dpi=900,bbox_inches='tight')
plt.show()

# 5 Figure 6d

In [ ]:
my_genes=[
    "PGF","MAP1LC3B","GDF15", "SERPINE1","LEP","PSG1", "TRPV6","LAIR2","MCAM","CCNB1","RRM2","ERVFRD-1","BCAM", "DLK1","PEG10","MCM7","FCGR3B","CD3E","CD68", "PECAM1","HBZ","KDR","COL5A2","THY1"
]
sc.pl.dotplot(
    placenta,
    var_names=my_genes,
    groupby='leiden_hvg_sub',  # 你的分组列名
    standard_scale='var',  # 标准化
    #vmax=2, 
    cmap="Reds",
    figsize=(10, 6),
    dendrogram=True,
    save='placenta_leiden_sub_normalize.pdf'
)

# 6 Figure 6f

In [ ]:
import pandas as pd
import os
from pathlib import Path


folder_path = r"P:\PI\PI_Chuva_de_Sousa_Lopes\susana\Fu\Project\20_Xenium 5k embryo\From Jamilla\DASH_APP\10_2025_12_5_whole_placenta\cell_coordination_ line_to_surface"


excel_files = []
for file in os.listdir(folder_path):
    if file.startswith("coordinate_mapping_all_together") and file.endswith((".xlsx", ".xls", ".csv")):
        excel_files.append(os.path.join(folder_path, file))


for f in excel_files:
    print(f"  - {os.path.basename(f)}")


all_data = []

for file in sorted(excel_files):
    

    if file.endswith('.csv'):
        df = pd.read_csv(file)
    else:
        df = pd.read_excel(file)
    
    
    all_data.append(df)


merged_df = pd.concat(all_data, ignore_index=True)


output_file = os.path.join(folder_path, "coordinate_mapping_merged.csv")
merged_df.to_csv(output_file, index=False)



In [ ]:
import scanpy as sc
import pandas as pd
from pathlib import Path

folder_path = r"P:\PI\PI_Chuva_de_Sousa_Lopes\susana\Fu\Project\20_Xenium 5k embryo\From Jamilla\DASH_APP\10_2025_12_5_whole_placenta"
h5ad_file = Path(folder_path) / "whole_placenta_updata_hvg_leiden_filter_3st.h5ad"
placenta= sc.read_h5ad(h5ad_file)
placenta

In [ ]:
coord_mapping=merged_df
coord_mapping["cell_id1"]=coord_mapping["id"]
coord_mapping_selected = coord_mapping[['cell_id1', 'new_x', 'new_y', 'new_z']].rename(columns={
    'new_x': 'DV',
    'new_y': 'LR',
    'new_z': 'AP'
})

placenta.obs = placenta.obs.merge(coord_mapping_selected, on='cell_id1', how='left')

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from scipy.interpolate import make_interp_spline
from scipy.stats import gaussian_kde
from scipy.ndimage import gaussian_filter1d

def _intersections_on_grid(x, y1, y2):
    """ x """
    d = y1 - y2
    s = np.sign(d)
    idx = np.where(np.diff(s) != 0)[0]    # 
    xs = []
    for i in idx:
        x0, x1 = x[i], x[i+1]
        d0, d1 = d[i], d[i+1]
        if np.isfinite(d0) and np.isfinite(d1) and (d1 != d0):
            xs.append(x0 - d0 * (x1 - x0) / (d1 - d0))
    return xs

def plot_binned_density(
    adata,
    coord_col='AP',                 #  ( 'AP', 'DV', 'LR')
    group_col='leiden_1',           #  ( 'leiden_1', 'cell_type')
    n_bins=50,
    palette=None,
    figsize=(15, 6),
    xlim=None,
    ylim=None,
    fill=True,
    fill_alpha=0.5,                 # 0.5
    linewidth=2.5,
    smooth=0,
    show_legend=True,
    vertical_lines=None,
    vline_color='blue',
    vline_style='--',
    vline_width=1.5,
    vline_alpha=0.8,
    density_method='pdf',
    pairs_to_draw=None,
    post_smooth_sigma=0,
    dashed_groups=None,             #  ['1', '3', '5']
    secondary_y_groups=None,        # Y ['10', '11']
    secondary_ylim=None,            # Y
    x_tick_interval=100,            # X100
    primary_y_groups=None,          # 🆕 Y ['0', '1', '2']None
):
    """
     (coord_col)  bins  (group_col) 
    
    :
    - fill_alpha: float,  (0-1)0.5
    - dashed_groups: list of str, 
    - secondary_y_groups: list of str, Y
    - secondary_ylim: tuple, Y (ymin, ymax)
    - x_tick_interval: int or float, X100
    - primary_y_groups: list of str, YNonesecondary
    """

    # —— 1) 
    plot_df = adata.obs.copy()
    
    # 
    if coord_col not in plot_df.columns:
        raise KeyError(f"obs  '{coord_col}'")
    if group_col not in plot_df.columns:
        raise KeyError(f"obs  '{group_col}'")
    
    # 
    plot_df[coord_col] = pd.to_numeric(plot_df[coord_col], errors='coerce')
    plot_df = plot_df.dropna(subset=[coord_col, group_col])
    
    if plot_df.empty:
        print("Warning: No valid data found.")
        return None, {}

    #  cluster  1 vs '1' 
    plot_df[group_col] = plot_df[group_col].astype(str)
    all_groups = sorted(plot_df[group_col].unique(), key=lambda x: (len(x), x))

    # 
    dashed_groups = [str(g) for g in dashed_groups] if dashed_groups else []
    secondary_y_groups = [str(g) for g in secondary_y_groups] if secondary_y_groups else []
    primary_y_groups = [str(g) for g in primary_y_groups] if primary_y_groups else None

    # 🆕 
    if primary_y_groups is None and secondary_y_groups:
        # secondarysecondary
        groups_to_plot = all_groups
    elif primary_y_groups is not None:
        # primary_y_groupsprimary + secondary
        groups_to_plot = list(set(primary_y_groups) | set(secondary_y_groups))
        # 
        groups_to_plot = [g for g in all_groups if g in groups_to_plot]
    else:
        # 
        groups_to_plot = all_groups

    # —— 2)  bin
    coord_min, coord_max = plot_df[coord_col].min(), plot_df[coord_col].max()
    bins = np.linspace(coord_min, coord_max, n_bins + 1)
    bin_centers = (bins[:-1] + bins[1:]) / 2
    bin_width = bins[1] - bins[0]

    # —— 3)  group //
    densities = {}
    for cl in groups_to_plot:  # 🆕 
        #  group_col  coord_col
        vals = plot_df.loc[plot_df[group_col] == cl, coord_col].to_numpy()

        if density_method == 'kde':
            try:
                kde = gaussian_kde(vals)
                density = kde(bin_centers)
            except Exception:
                # 
                hist, _ = np.histogram(vals, bins=bins)
                density = hist / (len(vals) * bin_width if len(vals) > 0 else 1)
        else:
            hist, _ = np.histogram(vals, bins=bins)
            if density_method == 'pdf':
                density = hist / (len(vals) * bin_width if len(vals) > 0 else 1)
            elif density_method == 'frequency':
                density = hist / bin_width
            elif density_method == 'relative':
                density = hist / (len(vals) if len(vals) > 0 else 1)
            elif density_method == 'count':
                density = hist.astype(float)
            elif density_method == 'normalized':
                m = hist.max() if hist.size > 0 else 0
                density = hist / (m + 1e-10)
            else:
                raise ValueError(f": {density_method}")

        densities[cl] = density

    # —— 3.5)  + 
    x_dense = np.linspace(bin_centers.min(), bin_centers.max(), n_bins * 20)
    y_dense = {}
    for cl in groups_to_plot:  # 🆕 
        y = densities[cl]
        y_interp = np.interp(x_dense, bin_centers, y)
        if post_smooth_sigma and post_smooth_sigma > 0 and density_method != 'kde':
            y_interp = gaussian_filter1d(y_interp, sigma=post_smooth_sigma)
        y_dense[cl] = y_interp

    # —— 3.6) 
    intersections = {}  # {(cl1, cl2): [x1, x2, ...]}
    for i, c1 in enumerate(groups_to_plot):
        for j in range(i+1, len(groups_to_plot)):
            c2 = groups_to_plot[j]
            xs = _intersections_on_grid(x_dense, y_dense[c1], y_dense[c2])
            intersections[(c1, c2)] = xs

    # —— 4)  - Y
    fig, ax = plt.subplots(figsize=figsize)
    
    # Y
    ax2 = None
    if secondary_y_groups:
        ax2 = ax.twinx()

    # 🆕 
    if primary_y_groups is not None:
        # primary_y_groups
        primary_groups = [g for g in groups_to_plot if g in primary_y_groups and g not in secondary_y_groups]
    else:
        # secondary
        primary_groups = [g for g in groups_to_plot if g not in secondary_y_groups]
    
    secondary_groups = [g for g in groups_to_plot if g in secondary_y_groups]

    # Y
    for cl in primary_groups:
        color = palette.get(cl, None) if palette else None
        y = densities[cl]
        
        # 
        linestyle = '--' if cl in dashed_groups else '-'

        if smooth > 0 and density_method != 'kde':
            try:
                x_smooth = np.linspace(bin_centers.min(), bin_centers.max(), n_bins * 10)
                k = min(int(smooth), len(bin_centers) - 1, 5)
                spl = make_interp_spline(bin_centers, y, k=k)
                y_s = np.maximum(spl(x_smooth), 0)
                if fill:
                    ax.fill_between(x_smooth, y_s, alpha=fill_alpha, color=color, label=f'{group_col} {cl}')
                ax.plot(x_smooth, y_s, linewidth=linewidth, color=color,
                       linestyle=linestyle,
                       label=f'{group_col} {cl}' if not fill else '')
            except Exception:
                if fill:
                    ax.fill_between(bin_centers, y, alpha=fill_alpha, color=color, label=f'{group_col} {cl}')
                ax.plot(bin_centers, y, linewidth=linewidth, color=color,
                       linestyle=linestyle,
                       label=f'{group_col} {cl}' if not fill else '')
        else:
            if fill:
                ax.fill_between(bin_centers, y, alpha=fill_alpha, color=color, label=f'{group_col} {cl}')
            ax.plot(bin_centers, y, linewidth=linewidth, color=color,
                   linestyle=linestyle,
                   label=f'{group_col} {cl}' if not fill else '')

    # Y
    if ax2:
        for cl in secondary_groups:
            color = palette.get(cl, None) if palette else None
            y = densities[cl]
            
            linestyle = '--' if cl in dashed_groups else '-'

            if smooth > 0 and density_method != 'kde':
                try:
                    x_smooth = np.linspace(bin_centers.min(), bin_centers.max(), n_bins * 10)
                    k = min(int(smooth), len(bin_centers) - 1, 5)
                    spl = make_interp_spline(bin_centers, y, k=k)
                    y_s = np.maximum(spl(x_smooth), 0)
                    if fill:
                        ax2.fill_between(x_smooth, y_s, alpha=fill_alpha, color=color, label=f'{group_col} {cl}')
                    ax2.plot(x_smooth, y_s, linewidth=linewidth, color=color,
                            linestyle=linestyle,
                            label=f'{group_col} {cl}' if not fill else '')
                except Exception:
                    if fill:
                        ax2.fill_between(bin_centers, y, alpha=fill_alpha, color=color, label=f'{group_col} {cl}')
                    ax2.plot(bin_centers, y, linewidth=linewidth, color=color,
                            linestyle=linestyle,
                            label=f'{group_col} {cl}' if not fill else '')
            else:
                if fill:
                    ax2.fill_between(bin_centers, y, alpha=fill_alpha, color=color, label=f'{group_col} {cl}')
                ax2.plot(bin_centers, y, linewidth=linewidth, color=color,
                        linestyle=linestyle,
                        label=f'{group_col} {cl}' if not fill else '')

    # 
    if vertical_lines is not None:
        for xv in vertical_lines:
            ax.axvline(xv, color=vline_color, linestyle=vline_style,
                        linewidth=vline_width, alpha=vline_alpha, zorder=10)

    #  pair 
    if pairs_to_draw:
        for p in pairs_to_draw:
            c1, c2 = map(str, p)  # 
            xs = intersections.get((c1, c2), []) + intersections.get((c2, c1), [])
            for xv in xs:
                ax.axvline(xv, color='k', linestyle=':', lw=1.2, alpha=0.9, zorder=11)

    # 
    if xlim: ax.set_xlim(xlim)
    if ylim: ax.set_ylim(ylim)
    if secondary_ylim and ax2: ax2.set_ylim(secondary_ylim)
    
    ax.set_xlabel(f"{coord_col} Position", fontsize=12)

    ylabel_dict = {
        'pdf': 'Probability Density',
        'frequency': 'Frequency Density',
        'relative': 'Relative Frequency',
        'count': 'Cell Count',
        'kde': 'Density (KDE)',
        'normalized': 'Normalized Density'
    }
    ax.set_ylabel(ylabel_dict.get(density_method, 'Density'), fontsize=12)
    
    # Y
    if ax2:
        ax2.set_ylabel(f'{ylabel_dict.get(density_method, "Density")} (Secondary)', 
                       fontsize=12)
    
    ax.set_title(f'{coord_col} Distribution of {group_col} Clusters ({density_method.upper()} Method)',
                 fontsize=18, fontweight='bold', pad=20)

    ax.xaxis.set_major_locator(mticker.MultipleLocator(x_tick_interval))
    ax.tick_params(axis='x', rotation=90, labelsize=10)
    
    #  - Y
    if show_legend:
        lines1, labels1 = ax.get_legend_handles_labels()
        if ax2:
            lines2, labels2 = ax2.get_legend_handles_labels()
            ax.legend(lines1 + lines2, labels1 + labels2, 
                     bbox_to_anchor=(1.15, 1), loc='upper left')
        else:
            ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    
    ax.margins(x=0, y=0.02)
    plt.tight_layout()

    # 
    info = {
        "bin_centers": bin_centers,
        "densities": densities,      #  bin 
        "x_dense": x_dense,
        "y_dense": y_dense,          # 
        "intersections": intersections,
        "ax2": ax2,                  # 
        "primary_groups": primary_groups,  # 🆕 
        "secondary_groups": secondary_groups  # 🆕 
    }
    return fig, info

In [ ]:
import scanpy as sc
import pandas as pd
from pathlib import Path

folder_path = r"P:\PI\PI_Chuva_de_Sousa_Lopes\susana\Fu\Project\20_Xenium 5k embryo\analysis\2st_2025_4_9\backup_adata\QC_10_50_2st"
h5ad_file = Path(folder_path) / "final_mtx_v3_Fu_adjustedz_subcl.h5ad"
adata= sc.read_h5ad(h5ad_file)
adata

In [ ]:

adata.obs['leiden_hvg_1'] = None
adata.obs['source'] = None


embryo_mapping = embryo.obs['leiden_hvg_1']
mask_embryo = adata.obs.index.isin(embryo_mapping.index)
adata.obs.loc[mask_embryo, 'leiden_hvg_1'] = embryo_mapping.reindex(adata.obs.index[mask_embryo]).values
adata.obs.loc[mask_embryo, 'source'] = 'embryo'

placenta_mapping = placenta.obs['leiden_hvg_1']
mask_placenta = adata.obs.index.isin(placenta_mapping.index)
adata.obs.loc[mask_placenta, 'leiden_hvg_1'] = placenta_mapping.reindex(adata.obs.index[mask_placenta]).values
adata.obs.loc[mask_placenta, 'source'] = 'placenta'


In [ ]:
import numpy as np


adata.obs['cluster'] = 'other'

adata.obs.loc[(adata.obs['source'] == 'embryo') & (adata.obs['leiden_hvg_1'] == '7'), 'cluster'] = 'blood'
adata.obs.loc[(adata.obs['source'] == 'placenta') & (adata.obs['leiden_hvg_1'] == '20'), 'cluster'] = 'blood'


adata.obs.loc[(adata.obs['source'] == 'embryo') & (adata.obs['leiden_hvg_1'].isin(['9', '24'])), 'cluster'] = 'EC'
adata.obs.loc[(adata.obs['source'] == 'placenta') & (adata.obs['leiden_hvg_1'] == '10'), 'cluster'] = 'EC'


adata.obs.loc[(adata.obs['source'] == 'embryo') & (adata.obs['leiden_hvg_1'].isin(['13', '26'])), 'cluster'] = 'IM'
adata.obs.loc[(adata.obs['source'] == 'placenta') & (adata.obs['leiden_hvg_1'] == '22'), 'cluster'] = 'IM'


adata.obs.loc[(adata.obs['source'] == 'embryo') & (adata.obs['leiden_hvg_1'] == '21'), 'cluster'] = 'NC'


adata.obs.loc[(adata.obs['source'] == 'placenta') & (adata.obs['leiden_hvg_1'] == '1'), 'cluster'] = 'FB'


adata.obs.loc[(adata.obs['source'] == 'placenta') & (adata.obs['leiden_hvg_1'] == '14'), 'cluster'] = 'PER'



In [ ]:

adata.obs['leiden_hvg_sub'] = None


embryo_mapping = embryo.obs['leiden_hvg_sub']
mask_embryo = adata.obs.index.isin(embryo_mapping.index)
adata.obs.loc[mask_embryo, 'leiden_hvg_sub'] = embryo_mapping.reindex(adata.obs.index[mask_embryo]).values


placenta_mapping = placenta.obs['leiden_hvg_sub']
mask_placenta = adata.obs.index.isin(placenta_mapping.index)
adata.obs.loc[mask_placenta, 'leiden_hvg_sub'] = placenta_mapping.reindex(adata.obs.index[mask_placenta]).values

adata_filtered = adata[~adata.obs['cluster'].isin(['other', 'FB'])].copy()


adata.obs['leiden_hvg_sub'] = adata.obs['leiden_hvg_sub'].astype(str)

mask_placenta = adata.obs['source'] == 'placenta'
adata.obs.loc[mask_placenta, 'leiden_hvg_sub'] = 'p' + adata.obs.loc[mask_placenta, 'leiden_hvg_sub']


In [ ]:
import pandas as pd

df = pd.read_csv(r"P:\PI\PI_Chuva_de_Sousa_Lopes\susana\Fu\Project\20_Xenium 5k embryo\From Jamilla\DASH_APP\13-2026-02-19_Figure5\selected_cells_2.csv")

In [ ]:
ec.obs.loc[ec.obs['source'] == 'BS', 'source'] = 'embryo'
adata.obs.loc[adata.obs['source'] == 'BS', 'source'] = 'embryo'
adata_filtered.obs.loc[adata_filtered.obs['source'] == 'BS', 'source'] = 'embryo'



adata.obs['source'] = adata.obs['source'].astype(str)


matching_cells = adata.obs.index.isin(df['cell_id'])
adata.obs.loc[matching_cells, 'source'] = 'BS'




adata_filtered.obs['source'] = adata_filtered.obs['source'].astype(str)


matching_cells = adata_filtered.obs.index.isin(df['cell_id'])
adata_filtered.obs.loc[matching_cells, 'source'] = 'BS'



In [ ]:
adata_filtered.write_h5ad(r"P:\PI\PI_Chuva_de_Sousa_Lopes\susana\Fu\Project\20_Xenium 5k embryo\paper\3st_2025_1_27\Figure5\adata\EC_IM_BL_NC.h5ad")

In [ ]:
keep = np.array(["EC"], dtype=str)

ec = adata_filtered[adata_filtered.obs["cluster"].isin(keep)].copy()

In [ ]:
ec.layers['counts_raw'] = ec.X.copy()

sc.pp.normalize_total(ec, inplace=True)
sc.pp.log1p(ec)


sc.pp.highly_variable_genes(ec, n_top_genes=2000)


ec.raw = ec.copy()


sc.pp.scale(ec, max_value=10)


sc.tl.pca(ec, use_highly_variable=True)


sc.pp.neighbors(ec)


sc.tl.umap(ec, min_dist=0.01, spread=2)


sc.tl.leiden(ec, resolution=0.5, key_added='leiden_sub')

print(f"ec (n={ec.n_obs}) processing complete.")
print(f"Total genes retained: {ec.n_vars}")
print(f"Highly variable genes used for analysis: {ec.var.highly_variable.sum()}")

In [ ]:
sc.tl.rank_genes_groups(ec, 
                        groupby="leiden_hvg_sub", 
                        method="wilcoxon", 
                        pts=True,
                        )

In [ ]:
deg_df = sc.get.rank_genes_groups_df(ec, group=None)

deg_df.to_excel(r"P:\PI\PI_Chuva_de_Sousa_Lopes\susana\Fu\Project\20_Xenium 5k embryo\paper\3st_2025_1_27\Figure5\deg\DEG_leiden_hvg_sub_ec.xlsx", index=True)



In [ ]:
sc.tl.dendrogram(ec, groupby="leiden_hvg_sub", linkage_method="ward",cor_method='pearson')  #'pearson', 'spearman', 'kendall',

sc.pl.dendrogram(ec, groupby="leiden_hvg_sub")



In [ ]:
custom_palette = {

    'e9_0': '#fb34cd',  
    'e9_1': '#e5e022',  
    'e9_2': '#10686f',  
    'e9_3': '#ffa5aa',  
    'e9_4': '#C1A029',  
    'e9_5': '#ff002a', 
    

    'e24_0': '#0066FF',
    'e24_1': '#FF4500', 
    'e24_2': '#FFD700',  
    
   'p10_0': '#00FFFF',  
   'p10_1': '#7FFF00',  
   'p10_2': '#8A2BE2', 
   'p10_3': '#0066FF', 
}

In [ ]:

tsne_coords =ec.obsm['X_umap']


leiden_clusters = ec.obs['leiden_hvg_sub']

import pandas as pd

tsne_df = pd.DataFrame(tsne_coords, columns=['UMAP_1', 'UMAP_2'])
tsne_df['leiden_hvg_sub'] = leiden_clusters.values

import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(6, 4))
sns.scatterplot(data=tsne_df, x='UMAP_1', y='UMAP_2', hue='leiden_hvg_sub', palette=custom_palette, s=5, edgecolor=None)
plt.title("UMAP Plot with Leiden Clusters")
plt.xlabel("UMAP1")
plt.ylabel("UMAP2")
plt.legend(title="Leiden Cluster", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.savefig(r'P:\PI\PI_Chuva_de_Sousa_Lopes\susana\Fu\Project\20_Xenium 5k embryo\paper\3st_2025_1_27\Figure5\umap\umap_EC.png', format="png",dpi=900,bbox_inches='tight')
plt.show()

In [ ]:
my_genes=[
     "KDR","BMPER", "SLC9A3R1","SGK1","SOX17", "HMGA1","THBS1","CARTPT", "GJA5","PAGE4","HGF", "IGF2",
]
sc.pl.dotplot(
    ec,
    var_names=my_genes,
    groupby='leiden_hvg_sub',  # 你的分组列名
    standard_scale='var',  # 标准化
    #vmax=2, 
    cmap="Reds",
    figsize=(6, 3.5),
    dendrogram=True,
    save='ec_leiden_sub_normalize.pdf'
)

In [ ]:
keep = np.array(["blood"], dtype=str)

bl = adata_filtered[adata_filtered.obs["cluster"].isin(keep)].copy()

In [ ]:
bl.layers['counts_raw'] = bl.X.copy()

sc.pp.normalize_total(bl, inplace=True)
sc.pp.log1p(bl)

sc.pp.highly_variable_genes(bl, n_top_genes=2000)

bl.raw = bl.copy()

sc.pp.scale(bl, max_value=10)

sc.tl.pca(bl, use_highly_variable=True)

sc.pp.neighbors(bl)

sc.tl.umap(bl, min_dist=0.01, spread=2)

sc.tl.leiden(bl, resolution=0.5, key_added='leiden_sub')
print(f"bl (n={bl.n_obs}) processing complete.")
print(f"Total genes retained: {bl.n_vars}")
print(f"Highly variable genes used for analysis: {bl.var.highly_variable.sum()}")

In [ ]:
sc.tl.rank_genes_groups(bl, 
                        groupby="leiden_hvg_sub", 
                        method="wilcoxon", 
                        pts=True,
                        )

In [ ]:
deg_df = sc.get.rank_genes_groups_df(bl, group=None)

deg_df.to_excel(r"P:\PI\PI_Chuva_de_Sousa_Lopes\susana\Fu\Project\20_Xenium 5k embryo\paper\3st_2025_1_27\Figure5\deg\DEG_leiden_hvg_sub_bl.xlsx", index=True)



In [ ]:
sc.tl.dendrogram(bl, groupby="leiden_sub", linkage_method="ward",cor_method='pearson')  #'pearson', 'spearman', 'kendall',

sc.pl.dendrogram(bl, groupby="leiden_sub")

In [ ]:

tsne_coords =bl.obsm['X_umap']


leiden_clusters = bl.obs['leiden_hvg_sub']

import pandas as pd

tsne_df = pd.DataFrame(tsne_coords, columns=['UMAP_1', 'UMAP_2'])
tsne_df['leiden_hvg_sub'] = leiden_clusters.values

import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(6, 4))
sns.scatterplot(data=tsne_df, x='UMAP_1', y='UMAP_2', hue='leiden_hvg_sub', palette=custom_palette, s=5, edgecolor=None)
plt.title("UMAP Plot with Leiden Clusters")
plt.xlabel("UMAP1")
plt.ylabel("UMAP2")
plt.legend(title="Leiden Cluster", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.savefig(r'P:\PI\PI_Chuva_de_Sousa_Lopes\susana\Fu\Project\20_Xenium 5k embryo\paper\3st_2025_1_27\Figure5\umap\umap_BL.png', format="png",dpi=900,bbox_inches='tight')
plt.show()

In [ ]:
my_genes=[
     "HBZ","KDR", "FRZB","BTK","KEL","CCNB1"
]
sc.pl.dotplot(
    bl,
    var_names=my_genes,
    groupby='leiden_sub',  # 你的分组列名
    standard_scale='var',  # 标准化
    #vmax=2, 
    cmap="Reds",
    figsize=(4.2, 2),
    dendrogram=True,
    save='bl_leiden_sub_normalize.pdf'
)

In [ ]:

moran_df = adata_detailed.uns['moranI'].copy()
is_significant = moran_df['pval_norm_fdr_bh'] < 0.05


is_paracrine = moran_df['sender'] != moran_df['receiver']


filtered_results = moran_df[is_significant & is_paracrine & is_positive].copy()


genes = filtered_results.index.tolist()


In [ ]:
sub_cluster_palette = {
    "e6_2":  "#fb34cd",  
    "e6_3":  "#FE664D",  
    "e6_6":  "#009203",  
    "e6_7":  "#ff002a",  
    
    "e8":  "#FF6347", 
    "e8_1":  "#8e0119",  
    "e8_2":  "#C1A029",  
    "e8_4":  "#00CED1",  
    
    "e11":   "#565DFD",  
    
    "e16_0": "#565DFD",  
    "e16_1": "#FF0000",  
    "e16_2": "#FF1493", 
    
    "e17_0": "#e5e022",  
    "e17_1": "#00FFFF", 
    "e17_2": "#4fa9ff",  
    "e17_3": "#1068be",  
    "e17_4": "#565DFD", 
    "e17_5": "#ffa5aa",  
    "e17_6": "#ADFF2F", 
    "e17_6.0": "#fb34cd",
    "e17_6.1": "#FE664D",
    "e17_6.2": "#009203",
    "e17_7": "#66c102",  
    "e20_0": "#66c102",
    "e20_1": "#8e0119", 
    "e20_2": "#10686f",

  
    "e25_0": "#7FFF00", 
    "e25_1": "#9f50f9",
    "e25_2": "#32a88e" , 
    
   "e22_0": "#AB76AE",  
    "e22_1": "#6a32a8", 
    "e22_2": "#FF1493", 
    "e22_3": "#10686f",  
}

In [ ]:

wnt_genes = [g for g in genes if 'WNT5A_FZD3' in g or 'SHH_PTCH1' in g or 'BMP7_BMPR1B' in g]
norm_df = create_advanced_lr_heatmap(
    adata_detailed, 
    gene_list=genes, 
    axis_col='AP',
    sender_col='sender',     
    receiver_col='receiver',
    cluster_colors=sub_cluster_palette, 
    n_bins=40,
    figsize=(6, 4),        
    order_method='peak_grouped',
    binning='linear',
    vlines=[280,480,840, 2100,2600],
    smooth='gaussian', smooth_sigma=1.5,
    highlight_genes=wnt_genes, 
    highlight_label_position='right',
    vmin=-2, vmax=2,
    show_only_highlighted=True,
    #x_tick_interval=500,
    xlim=(0,3000),
    save_path=r'P:\PI\PI_Chuva_de_Sousa_Lopes\susana\Fu\Project\20_Xenium 5k embryo\paper\3st_2025_1_27\Figure4\cellnest\cellnest_GUT_DOSAL_niche_heatmap_along_ap_small_1.png'
)


In [ ]:
df.to_csv(r"P:\PI\PI_Chuva_de_Sousa_Lopes\susana\Fu\Project\20_Xenium 5k embryo\From Jamilla\DASH_APP\13-2026-02-19_Figure5\df.csv", index=False)

In [ ]:
keep = np.array(["IM"], dtype=str)

im = adata_filtered[adata_filtered.obs["cluster"].isin(keep)].copy()

In [ ]:
im.layers['counts_raw'] = im.X.copy()

sc.pp.normalize_total(im, inplace=True)
sc.pp.log1p(im)

sc.pp.highly_variable_genes(im, n_top_genes=2000)

im.raw = im.copy()

sc.pp.scale(im, max_value=10)

sc.tl.pca(im, use_highly_variable=True)

sc.pp.neighbors(im)

sc.tl.umap(im, min_dist=0.01, spread=2)

sc.tl.leiden(im, resolution=0.5, key_added='leiden_sub')
print(f"im (n={im.n_obs}) processing complete.")
print(f"Total genes retained: {im.n_vars}")
print(f"Highly variable genes used for analysis: {im.var.highly_variable.sum()}")

In [ ]:
sc.tl.rank_genes_groups(im, 
                        groupby="leiden_sub", 
                        method="wilcoxon", 
                        pts=True,
                        )

In [ ]:
deg_df = sc.get.rank_genes_groups_df(im, group=None)

deg_df.to_excel(r"P:\PI\PI_Chuva_de_Sousa_Lopes\susana\Fu\Project\20_Xenium 5k embryo\paper\3st_2025_1_27\Figure5\deg\DEG_leiden_hvg_sub_im.xlsx", index=True)



In [ ]:
sc.tl.dendrogram(im, groupby="leiden_sub", linkage_method="ward",cor_method='pearson')  #'pearson', 'spearman', 'kendall',

sc.pl.dendrogram(im, groupby="leiden_sub")

In [ ]:
custom_palette = {
    '0': '#00FF7F',  
    '1': '#FF1493',
    '2': '#FF8C00', 
    '3': '#4169E1', 
    '4': '#FF00FF',  
}

In [ ]:

tsne_coords =imc.obsm['X_umap']


leiden_clusters = imc.obs['leiden_sub']

import pandas as pd

tsne_df = pd.DataFrame(tsne_coords, columns=['UMAP_1', 'UMAP_2'])
tsne_df['leiden_sub'] = leiden_clusters.values

import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(6, 4))
sns.scatterplot(data=tsne_df, x='UMAP_1', y='UMAP_2', hue='leiden_sub', palette=custom_palette, s=5, edgecolor=None)
plt.title("UMAP Plot with Leiden Clusters")
plt.xlabel("UMAP1")
plt.ylabel("UMAP2")
plt.legend(title="Leiden Cluster", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.savefig(r'P:\PI\PI_Chuva_de_Sousa_Lopes\susana\Fu\Project\20_Xenium 5k embryo\paper\3st_2025_1_27\Figure5\umap\im\umap_im.png', format="png",dpi=900,bbox_inches='tight')
plt.show()

In [ ]:
my_genes= [
   "MRC1","CD68", "F13A1", 
    "CCNB1", "PCNA", "OLFML3"
]
sc.pl.dotplot(
    imc,
    var_names=my_genes,
    groupby='leiden_sub',  # 你的分组列名
    standard_scale='var',  # 标准化
    #vmax=2, 
    cmap="Reds",
    figsize=(4.5, 2),
    dendrogram=True,
    save='im_leiden_sub_normalize.pdf'
)

In [ ]:
keep = np.array(["NC"], dtype=str)

nc = adata_filtered[adata_filtered.obs["cluster"].isin(keep)].copy()

In [ ]:
nc.layers['counts_raw'] = nc.X.copy()

sc.pp.normalize_total(nc, inplace=True)
sc.pp.log1p(nc)

sc.pp.highly_variable_genes(nc, n_top_genes=2000)

nc.raw = nc.copy()

sc.pp.scale(nc, max_value=10)

sc.tl.pca(nc, use_highly_variable=True)

sc.pp.neighbors(nc)

sc.tl.umap(nc, min_dist=0.01, spread=2)

sc.tl.leiden(nc, resolution=0.5, key_added='leiden_sub')
print(f"nc (n={nc.n_obs}) processing complete.")
print(f"Total genes retained: {nc.n_vars}")
print(f"Highly variable genes used for analysis: {nc.var.highly_variable.sum()}")

In [ ]:
sc.tl.rank_genes_groups(nc, 
                        groupby="leiden_sub", 
                        method="wilcoxon", 
                        pts=True,
                        )

In [ ]:
deg_df = sc.get.rank_genes_groups_df(nc, group=None)

deg_df.to_excel(r"P:\PI\PI_Chuva_de_Sousa_Lopes\susana\Fu\Project\20_Xenium 5k embryo\paper\3st_2025_1_27\Figure5\deg\DEG_leiden_hvg_sub_nc.xlsx", index=True)

In [ ]:
sc.tl.dendrogram(nc, groupby="leiden_sub", linkage_method="ward",cor_method='pearson')  #'pearson', 'spearman', 'kendall',

sc.pl.dendrogram(nc, groupby="leiden_sub")

In [ ]:
custom_palette = {
   '0': '#D2691E',  
    '1': '#9ACD32',  
    '2': '#00BFFF',  
    '3': '#66CDAA', 
}

In [ ]:

tsne_coords =nc.obsm['X_umap']


leiden_clusters = nc.obs['leiden_sub']

import pandas as pd
tsne_df = pd.DataFrame(tsne_coords, columns=['UMAP_1', 'UMAP_2'])
tsne_df['leiden_sub'] = leiden_clusters.values

import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(6, 4))
sns.scatterplot(data=tsne_df, x='UMAP_1', y='UMAP_2', hue='leiden_sub', palette=custom_palette, s=8, edgecolor=None)
plt.title("UMAP Plot with Leiden Clusters")
plt.xlabel("UMAP1")
plt.ylabel("UMAP2")
plt.legend(title="Leiden Cluster", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.savefig(r'P:\PI\PI_Chuva_de_Sousa_Lopes\susana\Fu\Project\20_Xenium 5k embryo\paper\3st_2025_1_27\Figure5\umap\nc\umap_nc.png', format="png",dpi=900,bbox_inches='tight')
plt.show()

In [ ]:
my_genes= [
   "SOX10","MPZ","TWIST1", "IRX1", 
    "PCDH8", "SEMA3C"
]
sc.pl.dotplot(
    nc,
    var_names=my_genes,
    groupby='leiden_sub',  
    standard_scale='var',  
    #vmax=2, 
    cmap="Reds",
    figsize=(4.2, 2),
    dendrogram=True,
    save='nc_leiden_sub_normalize.pdf'
)

In [ ]:
vertical_lines = [0, 1500, 1900,]
fig = plot_binned_density(
    placenta, 
    n_bins=40,
    palette=custom_palette,
    figsize=(25, 8),
    smooth=2,  
    show_legend=False,
    linewidth=6,
    vertical_lines=vertical_lines,
    vline_width=6,
    density_method='count',
    coord_col='AP',
    group_col='leiden_hvg_1',
    #secondary_y_groups=["e17_2"], 
    #dashed_groups=["e17_2"],
    #secondary_ylim=(0, 300),
    #x_tick_interval=50,
    fill_alpha=0.5,
    xlim=(-300,2800),
    #ylim=(0,120),
    primary_y_groups=['1', '14', '10', '20','22']
)
#plt.savefig(r'P:\PI\PI_Chuva_de_Sousa_Lopes\susana\Fu\Project\20_Xenium 5k embryo\paper\2st_2025_10_07\Figure4\density_plot_placenta_fetal_cell.pdf', dpi=900, bbox_inches='tight')
plt.show()

In [ ]:
vertical_lines = [0, 1500, 1900,]
fig = plot_binned_density(
    placenta, 
    n_bins=40,
    palette=custom_palette,
    figsize=(35, 12),
    smooth=2,  # 轻微平滑
    show_legend=False,
    linewidth=6,
    #vertical_lines=vertical_lines,
    #vline_width=6,
    density_method='count',
    coord_col='AP',
    group_col='leiden_hvg_1',
    #secondary_y_groups=["e17_2"], 
    #dashed_groups=["e17_2"],
    #secondary_ylim=(0, 300),
    #x_tick_interval=50,
    fill_alpha=0.5,
    xlim=(-300,2800),
    #ylim=(0,120),
    primary_y_groups=['15', '21', '23', '16']
)
plt.savefig(r'P:\PI\PI_Chuva_de_Sousa_Lopes\susana\Fu\Project\20_Xenium 5k embryo\paper\2st_2025_10_07\Figure4\density_plot_placenta_mother_immunce.pdf', dpi=900, bbox_inches='tight')
plt.show()

In [ ]:
vertical_lines = [0, 1500, 1900,]
fig = plot_binned_density(
    placenta, 
    n_bins=40,
    palette=custom_palette,
    figsize=(35, 12),
    smooth=2,  
    show_legend=False,
    linewidth=6,
    #vertical_lines=vertical_lines,
    #vline_width=6,
    density_method='count',
    coord_col='AP',
    group_col='leiden_hvg_1',
    #secondary_y_groups=["e17_2"], 
    #dashed_groups=["e17_2"],
    #secondary_ylim=(0, 300),
    #x_tick_interval=50,
    fill_alpha=0.5,
    xlim=(-300,2800),
    #ylim=(0,120),
    primary_y_groups=['3', '6', '8', '18', '17', '0', '2',"19"]
)
plt.savefig(r'P:\PI\PI_Chuva_de_Sousa_Lopes\susana\Fu\Project\20_Xenium 5k embryo\paper\2st_2025_10_07\Figure4\density_plot_placenta_CCTB.pdf', dpi=900, bbox_inches='tight')
plt.show()

In [ ]:
vertical_lines = [0, 1500, 1900,]
fig = plot_binned_density(
    placenta, 
    n_bins=40,
    palette=custom_palette,
    figsize=(35, 12),
    smooth=2,  
    show_legend=False,
    linewidth=6,
    #vertical_lines=vertical_lines,
    #vline_width=6,
    density_method='count',
    coord_col='AP',
    group_col='leiden_hvg_1',
    #secondary_y_groups=["e17_2"], 
    #dashed_groups=["e17_2"],
    #secondary_ylim=(0, 300),
    #x_tick_interval=50,
    fill_alpha=0.5,
    xlim=(-300,2800),
    #ylim=(0,120),
    primary_y_groups=['4', '5', '7', '9', '11', '13','12']
)
plt.savefig(r'P:\PI\PI_Chuva_de_Sousa_Lopes\susana\Fu\Project\20_Xenium 5k embryo\paper\2st_2025_10_07\Figure4\density_plot_placenta_STB.pdf', dpi=900, bbox_inches='tight')
plt.show()

# 6 Figure 6g

In [ ]:
import anndata as ad
import squidpy as sq
import cellcharter as cc
import pandas as pd
import scanpy as sc
import scvi
import numpy as np
import matplotlib.pyplot as plt
scvi.settings.seed = 12345

In [ ]:
import scanpy as sc
import pandas as pd
from pathlib import Path
# 文件路径
folder_path = r"P:\PI\PI_Chuva_de_Sousa_Lopes\susana\Fu\Project\20_Xenium 5k embryo\analysis\2st_2025_4_9\backup_adata\QC_10_50_2st\placenta"
h5ad_file = Path(folder_path) / "whole_placenta_updata_hvg_leiden.h5ad"
placenta = sc.read_h5ad(h5ad_file)

In [ ]:

adata = placenta.copy()

# =========================
# 0) Input checks
# =========================
assert "counts_raw" in adata.layers.keys(), "layers['counts_raw'] not found"
spatial_key = "spatial_kde_aligned"   # 可改 "spatial" / "spatial_spline_aligned"
assert spatial_key in adata.obsm_keys(), f"{spatial_key} not in adata.obsm"

use_batch = False
batch_key = "batch"
if use_batch:
    assert batch_key in adata.obs.columns
    adata.obs[batch_key] = adata.obs[batch_key].astype("category")

# =========================
# 1) Filtering on raw counts
# =========================
adata.X = adata.layers["counts_raw"].copy()


# =========================
# 2) Save raw counts for scVI  (tutorial-style)
# =========================
adata.layers["counts"] = adata.X.copy()

# =========================
# 3) Normalize/log for QC & plotting (NOT for scVI)
# =========================
sc.pp.normalize_total(adata, target_sum=1e6)
sc.pp.log1p(adata)

# =========================
# 4) Dimensionality reduction (scVI)  
# =========================
setup_kwargs = dict(layer="counts")
if use_batch and adata.obs[batch_key].nunique() > 1:
    setup_kwargs["batch_key"] = batch_key

scvi.model.SCVI.setup_anndata(adata, **setup_kwargs)

model = scvi.model.SCVI(adata, n_latent=10)   
model.train(early_stopping=True, enable_progress_bar=True)

adata.obsm["X_scVI"] = model.get_latent_representation().astype(np.float32)

# =========================
# 5) Spatial graph (neighbors) for CellCharter
# =========================
sq.gr.spatial_neighbors(
    adata,
    coord_type="generic",
    delaunay=True,
    spatial_key=spatial_key,
    percentile=99,
    library_key=batch_key if (use_batch and adata.obs[batch_key].nunique() > 1) else None,
)
cc.gr.remove_long_links(adata)



In [ ]:
# =========================
# 6) CellCharter: neighborhood aggregation (core step)
# =========================
cc.gr.aggregate_neighbors(
    adata,
    n_layers=3,
    use_rep="X_scVI",
    out_key="X_cellcharter",
    sample_key=batch_key if (use_batch and adata.obs[batch_key].nunique() > 1) else None,
)

# =========================
# 7) CellCharter’s spatial clustering (AutoK -> Cluster)
# =========================
autok = cc.tl.ClusterAutoK(n_clusters=(2, 12), max_runs=10, random_state=0)
autok.fit(adata, use_rep="X_cellcharter")
cc.pl.autok_stability(autok)

K =7
clusterer = cc.tl.Cluster(n_clusters=K, random_state=0)
clusterer.fit(adata, use_rep="X_cellcharter")

print(adata.obs["cluster_cellcharter"].value_counts())

# =========================
# 8) Plot spatial domains
# =========================
sq.pl.spatial_scatter(
    adata,
    color="cluster_cellcharter",
    spatial_key=spatial_key,
    library_key=batch_key if (use_batch and adata.obs[batch_key].nunique() > 1) else None,
    size=15
)

In [ ]:
# =========================
# 7) CellCharter’s spatial clustering (AutoK -> Cluster)
# =========================
autok = cc.tl.ClusterAutoK(n_clusters=(2, 12), max_runs=10)
autok.fit(adata, use_rep="X_cellcharter")
cc.pl.autok_stability(autok)

In [ ]:
K = 7
clusterer = cc.tl.Cluster(n_clusters=K)
clusterer.fit(adata, use_rep="X_cellcharter")

cands = [a for a in dir(clusterer)
         if any(k in a.lower() for k in ["label", "cluster", "assign", "pred", "y_"])]
cands


In [ ]:
adata.obs["cluster_cellcharter"] = autok.predict(
    adata,
    use_rep="X_cellcharter"
).astype(str)

In [ ]:
import re
import pandas as pd


def sort_groups(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    df['group'] = df['group'].astype(str)

    s = df['group']

    m = s.str.extract(r'^e_(\d+)_(\d+)$')
    main = pd.to_numeric(m[0], errors='coerce')
    sub  = pd.to_numeric(m[1], errors='coerce')


    is_num = s.str.fullmatch(r'\d+')
    sub_num = pd.to_numeric(s.where(is_num), errors='coerce')


    BIG = 10**9
    main = main.fillna(BIG)
    sub  = sub.fillna(BIG)

    main = main.where(~is_num, 0)
    sub  = sub.where(~is_num, sub_num)


    out = (df.assign(_main=main, _sub=sub)
             .sort_values(['_main','_sub','group'], kind='mergesort')
             .drop(columns=['_main','_sub'])
             .reset_index(drop=True))
    return out
obs_data = adata.obs
selected_data = obs_data[obs_data['batch'] == "29381"][['cell_id', 'cluster_cellcharter']].rename(columns={'cluster_cellcharter': 'group'})
selected_data = sort_groups(selected_data)
selected_data.to_csv("For xenium explor visulization/QC_10/2st_HVG/whole_placenta_QC10_hvg_381_cellchart_section9.csv", index=False, sep=",")

In [ ]:
adata.write_h5ad(r"P:\PI\PI_Chuva_de_Sousa_Lopes\susana\Fu\Project\20_Xenium 5k embryo\analysis\2st_2025_4_9\backup_adata\QC_10_50_2st\\placenta\placenta_for_cellcharter.h5ad")

# 6 Figure 6h¶

In [ ]:
import scanpy as sc
import pandas as pd
from pathlib import Path
# 文件路径
h5ad_file = r"P:\PI\PI_Chuva_de_Sousa_Lopes\susana\Fu\Project\20_Xenium 5k embryo\From Jamilla\DASH_APP\10_2025_12_5_whole_placenta\whole_placenta_hvg_res_1.h5ad"

adata = sc.read_h5ad(h5ad_file)


In [ ]:

import pandas as pd


url = 'https://maayanlab.cloud/Enrichr/geneSetLibrary?mode=text&libraryName=MSigDB_Hallmark_2020'

def parse_gmt(url):
    df_list = []
    response = pd.read_csv(url, sep='\t', header=None, engine='python')
    for i, row in response.iterrows():
        pathway = row[0]
        genes = row[2:].dropna().astype(str).values
        for gene in genes:
            if gene:
                df_list.append([pathway, gene])
    return pd.DataFrame(df_list, columns=['geneset', 'genesymbol'])

net = parse_gmt(url)
net['geneset'] = 'HALLMARK_' + net['geneset'].astype(str).str.upper()
net['genesymbol'] = net['genesymbol'].str.upper()

In [ ]:
import scanpy as sc
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

xenium_genes = set(adata.var_names.str.upper())
score_columns = [col for col in adata.obs.columns if '_Score' in col]

coverage_dict = {}
for col in score_columns:
 
    # 'Tnf-Alpha Signaling Via Nf-Kb_Score' -> 'HALLMARK_TNF-ALPHA SIGNALING VIA NF-KB'
    pathway_clean = col.replace('_Score', '').upper().replace(' ', ' ')
    pathway_full = 'HALLMARK_' + pathway_clean
    

    ref_genes = set(net[net['geneset'] == pathway_full]['genesymbol'])
    
    if len(ref_genes) == 0:
 
        continue
    
    overlap_genes = ref_genes.intersection(xenium_genes)
    
    total = len(ref_genes)
    found = len(overlap_genes)
    coverage = (found / total * 100) if total > 0 else 0
    
    coverage_dict[col] = coverage



filtered_columns = [col for col, cov in coverage_dict.items() if cov >= 40]




df_scores = adata.obs[filtered_columns].copy()

adata_temp = sc.AnnData(X=df_scores.values)
adata_temp.obs = adata.obs[['niche']].copy()


clean_names = [col.replace('_Score', '') for col in filtered_columns]
adata_temp.var_names = clean_names 

sc.tl.dendrogram(adata_temp, groupby='niche')


df_mean = adata_temp.to_df()
df_mean['niche'] = adata_temp.obs['niche']
df_mean = df_mean.groupby('niche').mean()

g = sns.clustermap(df_mean.T, metric='correlation', method='complete', figsize=(1, 1))
plt.close() 

reordered_indices = g.dendrogram_row.reordered_ind
sorted_pathways = [df_mean.columns[i] for i in reordered_indices]


sc.pl.dotplot(
    adata_temp,
    var_names=sorted_pathways,
    groupby='niche',
    dendrogram=True,
    standard_scale='var',
    cmap='RdBu_r',
    dot_max=1.0,
    dot_min=0.2,
    figsize=(6, 12),
    swap_axes=True,
    colorbar_title='Relative Activity',
    size_title='Fraction of Cells',
    show=False
)

#save_path = r'P:\your_path\HALLMARK_pathway_filtered.pdf'
#plt.savefig(save_path, dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
sc.pl.dotplot(
    adata_temp,
    var_names=sorted_pathways,
    groupby='niche',
    dendrogram=True,
    standard_scale='var',
    cmap='RdBu_r',
    dot_max=1.0,
    dot_min=0.2,
    figsize=(4.5, 9),
    swap_axes=True,
    colorbar_title='Relative Activity',
    size_title='Fraction of Cells',
    show=False
)
save_path = "P:/PI/PI_Chuva_de_Sousa_Lopes/susana/Fu/Project/20_Xenium 5k embryo/paper/3st_2025_1_27/Figure6/pathway/placenta_pathway.pdf"
plt.savefig(save_path, dpi=300, bbox_inches='tight')
plt.show()